In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import time
from sklearn.preprocessing import StandardScaler

# 手机价格分类案例


# todo1: 提取数据集
def create_dataset():
    data = pd.read_csv("./手机价格预测.csv")
    # print(data)
    x, y = (
        data.iloc[:, :-1],
        data.iloc[:, -1],
    )  # 特征和标签, 特征是所有列除了最后一列, 标签是最后一列
    # print(x)
    # print(y)
    # 类型转换：特征值
    x = x.astype(np.float32)
    # print(type(x))

    # 数据集划分
    # random_state=88 保证每次划分结果一致，便于复现实验
    x_train, x_valid, y_train, y_valid = train_test_split(
        x, y, train_size=0.8, random_state=88
    )
    """优化①:数据标准化"""
    transfer = StandardScaler()
    x_train = transfer.fit_transform(x_train)
    x_valid = transfer.transform(x_valid)
    # print(x_train.values)
    # print(type(x_train.values))
    # 构建数据集,转换为pytorch的形式
    train_dataset = TensorDataset(
        torch.from_numpy(x_train), torch.tensor(y_train.values)
    )
    valid_dataset = TensorDataset(
        torch.from_numpy(x_valid), torch.tensor(y_valid.values)
    )
    # 返回结果
    # x_train.shape[1]: 一个样本中包含的特征数（样本数）
    # len(np.unique(y)): 输出类别数，输出类别为0，1，2，3
    return train_dataset, valid_dataset, x_train.shape[1], len(np.unique(y))


# todo2: 搭建神经网络
"""
构建全连接神经网络来进行手机价格分类，该网络主要由三个线性层来构建，使用relu激活函数。
网络共有 3 个全连接层, 具体信息如下:
第一层: 输入为维度为 20, 输出维度为: 128
第二层: 输入为维度为 128, 输出维度为: 256
第三层: 输入为维度为 256, 输出维度为: 4
"""


class PhonePriceModel(nn.Module):
    def __init__(self, inputSize, outSize):
        super().__init__()

        # 搭建神经网络
        self.linear1 = nn.Linear(inputSize, 128)
        self.linear2 = nn.Linear(128, 256)
        """优化②:增加网络深度"""
        # 3. 第三层: 输入为维度为 256, 输出维度为: 512
        self.linear3 = nn.Linear(256, 512)
        # 4. 第四层: 输入为维度为 512, 输出维度为: 128
        self.linear4 = nn.Linear(512, 128)
        # 5. 输出层: 输入为维度为 128, 输出维度为: 4
        self.linear5 = nn.Linear(128, outSize)

    def forward(self, x):
        # 隐藏层：加权求和 + 激活函数relu
        x = torch.relu(self.linear1(x))
        x = torch.relu(self.linear2(x))
        x = torch.relu(self.linear3(x))
        x = torch.relu(self.linear4(x))
        # 正常操作，使用softmax函数，将输出转换为概率分布，但是后面会使用CrossEntropyLoss损失函数，该函数会自动计算softmax，所以这里不需要手动计算softmax
        # x = torch.softmax(self.linear3(x), dim=1)
        return self.linear5(x)


# todo3: 模型训练
def train(train_dataset, inputSize, outSize):
    # 创建数据加载器，流程：数据->张量->数据集->数据加载器
    # 参1：数据集1600条，参2：每次训练16条，参3：是否打乱数据（训练集：打乱，测试集：不打乱）

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    # 创建神经网络模型
    model = PhonePriceModel(inputSize, outSize)
    # 定义损失函数，因为是多分类，这里使用多分类交叉熵函数
    criterion = nn.CrossEntropyLoss()
    # 创建优化器对象
    # optimizer = optim.SGD(model.parameters(), lr=0.01)
    """"优化③:使用Adam优化方法, 优化④:学习率变为1e-4"""
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    # 训练50轮
    # 每轮训练1600条，每批训练16条，共100批
    epochs = 50
    for epoch in range(epochs):
        start_time = time.time()  # 一轮训练开始时间
        # 记录每轮的总损失，以及批次数
        total_loss, batch_num = 0.0, 0
        for x, y in train_loader:
            # 切换模型（状态）
            model.train()
            y_pred = model(x)  # 预测值
            loss = criterion(y_pred, y)  # 计算损失值
            optimizer.zero_grad()  # 梯度清零
            loss.backward()  # 反向传播
            optimizer.step()  # 更新参数
            total_loss += loss.item()  # 累计损失值
            batch_num += 1  # 累计批次数
        # 本轮训练结束，打印训练信息
        print(
            f"轮次：{epoch + 1}, 损失值：{total_loss / batch_num:.4f}, 耗时：{time.time() - start_time:.2f}秒"
        )
    # 走到这里，模型多轮训练完成，保存模型（参数）
    # model.state_dict() 为模型的参数信息
    print(f"模型的参数信息:{model.state_dict()}")
    torch.save(model.state_dict(), "./model/phone_price_model.pth")


# todo: 模型评估
def evaluate(valid_dataset, inputSize, outSize):
    # 加载模型和训练好的网络参数
    model = PhonePriceModel(inputSize, outSize)
    # load_state_dict:将加载的参数字典应用到模型上
    # load:加载用来保存模型参数的文件
    model.load_state_dict(torch.load("./model/phone_price_model.pth"))
    # 每批训练8个样本
    data_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False)
    # 评估测试集
    correct = 0
    # 遍历测试集中的数据
    for x, y in data_loader:
        model.eval()  # 使用推理模式
        output = model(x)
        # print(output)
        # print(torch.softmax(output, dim=1))
        # 因为在训练时使用的交叉熵损失函数，包含了softmax，所以这边的output是未转换为概率分布的原始输出
        # argmax: 获取原始输出中，最大值对应的下标, 即类别编码0，1，2，3
        y_pred = torch.argmax(output, dim=1)
        # 预测结果和真实标签（shape：（4，））进行比较，相同则累加正确数
        correct += (y_pred == y).sum()
    # 准确率
    accuracy = correct.item() / len(valid_dataset)
    print(f"模型在验证集上的准确率: {accuracy:.4f}")


if __name__ == "__main__":
    train_dataset, valid_dataset, inputSize, outSize = create_dataset()
    # 打印数据集信息
    # print("训练数据集样本数:", len(train_dataset)) # (1600, 20)
    # print("验证数据集样本数:", len(valid_dataset)) # (400, 20)
    print("输入特征维度:", inputSize)  # 20
    print("输出类别数:", outSize)  # 4
    # model = PhonePriceModel(inputSize, outSize)
    # print(model.linear1)
    # 参数1：模型对象
    # 参数2：输入数据的形状，每批16条（行），每条20列
    # summary(model, input_size=(16, inputSize))

    # 训练模型
    # train(train_dataset, inputSize, outSize)

    # 评估模型
    evaluate(valid_dataset, inputSize, outSize)

输入特征维度: 20
输出类别数: 4
模型在验证集上的准确率: 0.9075
